# **Teknik Scraping Website**

## Alat

Berikut ini adalah library yang dibutuhkan untuk mengambil data berita pada detik.com

1. requests
2. beautifulSoap
3. trafilatura
4. pandas

In [26]:
import requests
from bs4 import BeautifulSoup
import trafilatura
import pandas as pd
import time
from tqdm import tqdm

In [47]:
def dapatkan_url_detik(url_dasar, target_jumlah=100):
    """Mengumpulkan URL artikel dengan format pagination query (?page=)."""
    url_list = []
    halaman = 1
    
    print(f"Mulai mencari di: {url_dasar}")
    
    with tqdm(total=target_jumlah, desc="Mengumpulkan URL") as pbar:
        while len(url_list) < target_jumlah:
            
            # PERBAIKAN FINAL: Menggunakan format query parameter ?page=
            url_indeks = f"{url_dasar}indeks?page={halaman}"
            
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
            }
            
            try:
                response = requests.get(url_indeks, headers=headers, timeout=15)
                
                if response.status_code != 200:
                    print(f"\n[Peringatan] Server menolak akses di {url_indeks} (Status: {response.status_code}).")
                    break
                    
                soup = BeautifulSoup(response.text, 'html.parser')
                
                artikel_elemen = soup.find_all('article')
                
                if not artikel_elemen:
                    print(f"\n[Peringatan] Tidak menemukan tag <article> di halaman {halaman}. Berhenti.")
                    break
                    
                for artikel in artikel_elemen:
                    link = None
                    
                    if 'i-link' in artikel.attrs:
                        link = artikel['i-link']
                    else:
                        tag_a = artikel.find('a')
                        if tag_a and 'href' in tag_a.attrs:
                            link = tag_a['href']
                    
                    # Validasi link
                    if link and url_dasar in link and 'foto' not in link and 'video' not in link:
                        if link not in url_list:
                            url_list.append(link)
                            pbar.update(1)
                            
                            if len(url_list) >= target_jumlah:
                                break
            except Exception as e:
                print(f"\n[Error] Gagal mengakses {url_indeks}: {e}")
                break
                
            halaman += 1
            if halaman > 50:
                print("\n[Batas] Mencapai batas maksimal halaman.")
                break
                
            time.sleep(1) 
            
    return url_list

In [32]:
def ekstrak_teks_trafilatura(url_list, kategori):
    """Mengekstrak teks utama menggunakan Trafilatura dengan progress bar."""
    data_ekstrak = []
    
    # Membungkus url_list dengan tqdm agar muncul progress bar saat ekstraksi
    for url in tqdm(url_list, desc=f"Ekstraksi teks {kategori}"):
        downloaded = trafilatura.fetch_url(url)
        if downloaded:
            teks_bersih = trafilatura.extract(downloaded)
            if teks_bersih:
                data_ekstrak.append({
                    'kategori': kategori,
                    'url': url,
                    'teks': teks_bersih
                })
        time.sleep(1.5)
        
    return data_ekstrak

In [48]:
url_tema_sport = "https://sport.detik.com/"
url_tema_finance = "https://finance.detik.com/"

In [49]:
url_sport = dapatkan_url_detik(url_tema_sport, 100)
url_finance = dapatkan_url_detik(url_tema_finance, 100)

Mulai mencari di: https://sport.detik.com/


Mengumpulkan URL: 100%|██████████| 100/100 [00:10<00:00,  9.99it/s]


Mulai mencari di: https://finance.detik.com/


Mengumpulkan URL: 100%|██████████| 100/100 [00:11<00:00,  8.86it/s]


In [50]:
data_sport = ekstrak_teks_trafilatura(url_sport, 'sport')
data_finance = ekstrak_teks_trafilatura(url_finance, 'finance')

Ekstraksi teks finance: 100%|██████████| 100/100 [03:02<00:00,  1.83s/it]


In [51]:
# 4. Simpan ke CSV MASING-MASING TEMA SECARA TERPISAH
# Menyimpan Dataset Sport
df_sport = pd.DataFrame(data_sport)
nama_file_sport = 'dataset_detik_sport.csv'
df_sport.to_csv(nama_file_sport, index=False, encoding='utf-8')
print(f"\nDataset Sport berhasil disimpan sebagai: {nama_file_sport}")

# Menyimpan Dataset Finance
df_finance = pd.DataFrame(data_finance)
nama_file_finance = 'dataset_detik_finance.csv'
df_finance.to_csv(nama_file_finance, index=False, encoding='utf-8')
print(f"Dataset Finance berhasil disimpan sebagai: {nama_file_finance}")


Dataset Sport berhasil disimpan sebagai: dataset_detik_sport.csv
Dataset Finance berhasil disimpan sebagai: dataset_detik_finance.csv
